In [9]:
!pip install PyMuPDF
!pip install tavily-python
!pip install sentence-transformers
!pip install groq



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.7 MB/s eta 0:00:00


**AGENT 1 :  CV Parser Agent**

In [2]:
import fitz  # PyMuPDF
import requests
import json
import re
from google.colab import files
from google.colab import userdata  # for storing Groq API key

def parse_cv(cv_file_path, model="llama-3.3-70b-versatile"):
    # Extract text from PDF
    doc = fitz.open(cv_file_path)
    cv_text = ""
    for page in doc:
        cv_text += page.get_text()
    print("CV text extracted (first 500 chars):")
    print(cv_text[:500], "...\n")
    print( "=========================================================")
    # Get Groq API key
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found. Please set it in Colab userdata.")
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    # Prepare the prompt
    data = {
        "model": model,
        "messages": [
            {   "role": "system",
                "content": "Extract structured CV info: name, email,location, experience_years"+
                           " skills,current_title,job_title, desired_role. Return JSON only."
            },
            {   "role": "user",
                "content": f"CV text: {cv_text}"
            }
        ], "temperature": 0.0
    }

    # Send request to Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers=headers,
        json=data
    )

    # Error handling
    if response.status_code != 200:
        print("Groq API Error:", response.status_code)
        print(response.text)
        return None

    result = response.json()
    try:
        cv_raw = result["choices"][0]["message"]["content"]
        # Remove ```json ... ``` code block if present
        cv_clean = re.sub(r"^```json\s*|\s*```$", "", cv_raw, flags=re.MULTILINE)
        cv_structured = json.loads(cv_clean)
        print("CV structured data extracted:")
        print(json.dumps(cv_structured, indent=2))
        return cv_structured
    except (KeyError, json.JSONDecodeError) as e:
        print("Failed to parse JSON from Groq response:", e)
        print("Raw response:", result)
        return None

**AGENT 2: Job Search Agent**

In [3]:
from tavily import TavilyClient
from google.colab import userdata

def search_jobs(job_title, job_location="", max_results=10):
    """
    Job Search Agent using Tavily
    Returns real job URLs, skips job boards
    """
    TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
    client = TavilyClient(TAVILY_API_KEY)

    query = f'{job_title} "{job_location}" "Apply for this job"'
    print(f"🔎 Searching Tavily for: {query}")

    response = client.search(query=query, max_results=max_results)

    jobs = []
    for r in response.get("results", []):
        url = r.get("url")
        # Skip job board pages
        if not url or any(domain in url for domain in ["indeed.com", "linkedin.com", "glassdoor", "totaljobs"]):
            continue

        jobs.append({
            "title": r.get("title"),
            "url": url,
            "content": r.get("content","")  # job description snippet
        })

    print(f"✅ Found {len(jobs)} real job pages")
    return jobs


**AGENT 3: Job Ranking Agent**

In [4]:
from sentence_transformers import SentenceTransformer, util

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def rank_jobs(cv_structured, jobs, top_k=5):
    """
    Rank jobs by CV match.

    cv_structured: dict from parse_cv()
    jobs: list of jobs with 'title', 'content'
    top_k: return top N jobs
    """
    # Convert skills list to string if needed
    skills = cv_structured.get("skills", [])
    if isinstance(skills, list):
        skills = " ".join(skills)

    cv_text = " ".join([
        cv_structured.get("desired_role", ""),
        skills,
        cv_structured.get("current_title", "")
    ])

    cv_embedding = model.encode(cv_text, convert_to_tensor=True)

    ranked_jobs = []

    for job in jobs:
        job_text = job.get("title","") + " " + job.get("content","")
        job_embedding = model.encode(job_text, convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(cv_embedding, job_embedding).item()

        job["similarity"] = similarity
        ranked_jobs.append(job)

    # Sort descending by similarity
    ranked_jobs.sort(key=lambda x: x["similarity"], reverse=True)

    return ranked_jobs[:top_k]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def print_jobs(jobs):


    if not jobs:
        print("❌ No jobs found")
        return

    for i, job in enumerate(jobs, start=1):
        print(f"{i}. {job.get('title')}")
        print(f"   🌐 URL: {job.get('url')}")
        print(f"   📨 Emails: {', '.join(job.get('emails', [])) or 'None found'}")
        print(f"   ✅ Apply Link: {job.get('url') or 'Not detected'}")
        print()


**AGENT 4: Cover Letter Generator Agent**

In [14]:
from groq import Groq
import os
from google.colab import userdata

def draft_cover_letter(cv_structured, top_job):

    # Get Groq API key
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found. Please set it in Colab userdata.")

    client = Groq(api_key=GROQ_API_KEY)

    prompt = f"""
Write a professional and concise cover letter.

Candidate details
Name: {cv_structured.get("name", "")}
Current title: {cv_structured.get("current_title", "")}
Years of experience: {cv_structured.get("experience_years", "")}
Skills: {", ".join(cv_structured.get("skills", []))}

Job details
Job title: {top_job.get("title", "")}
Company: {top_job.get("company", "the company")}

Rules
One page
Professional tone
No emojis
End with the candidate name
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert career coach and resume writer."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.6,
        max_tokens=600
    )

    return response.choices[0].message.content

In [ ]:
# Assume top_jobs is the ranked list from Agent 3
top_job = top_jobs[0]  # pick the highest-ranked job

cover_letter = draft_cover_letter(cv_structured, top_job)
print(cover_letter)



Dear Hiring Team at the company,

I am writing to express my interest in the position of DYNAMIC Definition & Meaning | Dictionary .com. 
With 2 years of experience as a MSc Student in Artificial Intelligence and skills in Java, Python, C, SQL, JavaScript, HTML/CSS, Git, Docker, VS Code, Visual Studio, Pandas, NumPy, Matplotlib, Snowpark, Spark, React, Node.js, Flask, NestJS, Angular,
I am confident in my ability to contribute effectively to your team.
I am particularly drawn to this opportunity because of the company's commitment to excellence and innovation in the industry.
I am eager to apply my experience and enthusiasm to help achieve your goals.

Please find my CV attached for your consideration. 
I look forward to the possibility of discussing how my background and skills align with this role.

Thank you for your time and consideration.

Best regards,
UJJWAL BARAL



**AGENT 5: Google Sheet Coordinator Agent**

In [8]:

def save_to_google_sheet(jobs, cover_letters, sheet_url):
    """
    Save jobs and generated cover letters to Google Sheet, using separate Role and Company.

    jobs: list of job dicts
    cover_letters: list of cover letter strings
    sheet_url: shareable Google Sheet URL
    """
    import gspread
    from google.colab import auth
    from google.auth import default

    # Authenticate in Colab
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    sh = gc.open_by_url(sheet_url)
    worksheet = sh.sheet1

    # Add headers
    worksheet.update('A1:G1', [["Role", "Company", "Location", "Apply Link", "Email", "Original Title", "Cover Letter"]])

    rows = []
    for job, cover_letter in zip(jobs, cover_letters):
        # Split title into role & company
        title = job.get("title", "")
        if " - " in title:
            role, company = title.rsplit(" - ", 1)
        else:
            role, company = title, ""

        rows.append([
            role.strip(),
            company.strip(),
            job.get("location", ""),
            job.get("url", ""),
            job.get("email", ""),
            title,
            cover_letter
        ])

    worksheet.update(f'A2:G{len(rows)+1}', rows)
    print(f"✅ Jobs & cover letters saved to Google Sheet: {sheet_url}")


In [ ]:
# # top_jobs and generated cover letters
# cover_letters = [draft_cover_letter(cv_structured, job) for job in top_jobs]

# # Your Google Sheet URL
# sheet_url = "https://docs.google.com/spreadsheets/d/1ypc167cmVDv3aOvkvq_jNT-Vfe0bOfyIAkBXLgz2Ypw/edit?usp=sharing"

# # Save to sheet
# save_to_google_sheet(top_jobs, cover_letters, sheet_url)


**Multi-Agent Job Application Pipeline**

In [16]:

from google.colab import files
print("🚀 Starting Agentic AI Job Application Pipeline...\n")
# Upload CV
uploaded = files.upload()
cv_file = list(uploaded.keys())[0]

#  Agent 1: Parse CV
cv_structured = parse_cv(cv_file)

# Get job title / location
job_title = input(
    f"\nEnter job title (default: {cv_structured.get('current_title','')}): "
) or cv_structured.get('desired_role','')

job_location = input(
    f"\nEnter city / Remote (default: {cv_structured.get('location','')}): "
) or cv_structured.get('location','')

# Agent 2: Search jobs
jobs = search_jobs(job_title, job_location)

print("\n🔍 Top Job Results:\n")
print_jobs(jobs)

if not jobs:
    print("\n❌ No jobs found. Stopping pipeline.")

else:
    # Agent 3: Rank jobs
    top_jobs = rank_jobs(cv_structured, jobs, top_k=5)
    print("\n🔍 Top jobs ranked by CV match:\n")
    print_jobs(top_jobs)

    # Agent 4: Generate cover letters
    cover_letters = []
    for job in top_jobs:
        cover_letters.append(draft_cover_letter(cv_structured, job))

# Print the top job's cover letter (first one)
if cover_letters:
    print("\n📄 Sample Cover letter of 1st Best Matched job:\n")
    print(cover_letters[0])

    # Agent 5: Save to Google Sheet
    sheet_url = input("Enter your Google Sheet URL: ")
    save_to_google_sheet(top_jobs, cover_letters, sheet_url)

    print("\n🎉 Pipeline complete! Check your Google Sheet.")


🚀 Starting Agentic AI Job Application Pipeline...



Saving UjjwalResume.pdf to UjjwalResume (2).pdf
CV text extracted (first 500 chars):
UJJWAL BARAL
Worthing BN11 1RE
baraluzwal@gmail.com
+447391453427
I am currently pursuing an MSc in Artificial Intelligence at the University of East London, where I am
deepening my knowledge in machine learning, data science, and intelligent systems. Prior to this, I gained
around two years of hands-on experience in the software industry, working on full-stack development
and contributing to various projects across the software development lifecycle. I am now seeking an
opportunity to apply my  ...

CV structured data extracted:
{
  "name": "UJJWAL BARAL",
  "email": "baraluzwal@gmail.com",
  "location": "Worthing BN11 1RE",
  "experience_years": 2,
  "skills": [
    "Java",
    "Python",
    "C",
    "SQL",
    "JavaScript",
    "HTML/CSS",
    "Git",
    "Docker",
    "VS Code",
    "Visual Studio",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "Snowpark",
    "Spark",
    "React",
    "Node.js",

/tmp/ipython-input-1444096597.py:22: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update('A1:G1', [["Role", "Company", "Location", "Apply Link", "Email", "Original Title", "Cover Letter"]])
/tmp/ipython-input-1444096597.py:43: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update(f'A2:G{len(rows)+1}', rows)


✅ Jobs & cover letters saved to Google Sheet: https://docs.google.com/spreadsheets/d/1ypc167cmVDv3aOvkvq_jNT-Vfe0bOfyIAkBXLgz2Ypw/edit?usp=sharing

🎉 Pipeline complete! Check your Google Sheet.


In [ ]:
# Rank the jobs
top_jobs = rank_jobs(cv_structured, jobs, top_k=3)

# Print similarity for each ranked job
print("Ranked Jobs with Similarity Scores:\n")
for job in top_jobs:
    print(f"Title: {job['title']}")
    print(f"Link: {job['url']:}")
    print(f"Similarity Score: {job['similarity']:.4f}")
    print()

Ranked Jobs with Similarity Scores:

Title: DYNAMIC Definition & Meaning | Dictionary .com
Link: https://www.dictionary.com/browse/dynamic
Similarity Score: 0.2901

Title: Dynamic - Definition, Meaning, Synonyms & Etymology
Link: https://www.betterwordsonline.com/dictionary/dynamic
Similarity Score: 0.2728

Title: Dynamic - Definition, Meaning & Synonyms | Vocabulary.com
Link: https://www.vocabulary.com/dictionary/dynamic
Similarity Score: 0.2352

